# `ingestion_GP.ipynb`
## Garis Panduan Perkhidmatan Penjagaan Luka Di Fasiliti Kesihatan Primer (MOH Malaysia 2019)

### Document profile

| Property | Detail |
|---|---|
| Language | Mixed — Malay administrative sections, English clinical content |
| Layout | Single-column (no column-sorting needed) |
| Total pages | 36 |
| **Useful pages** | 14, 15, 16, 18, 34 (see map below) |
| Tables | Treatment recommendation table spans pages 15–16 (8 wound types) |

### Page content map

| PDF page | Content | Action |
|---|---|---|
| 1–13 | Cover, TOC, foreword, admin, SOP, flowchart, committee members | **SKIP** |
| **14** | T.I.M.E. wound assessment framework | ✅ Extract |
| **15** | Wound algorithm + treatment table rows 1–2 | ✅ Extract (algorithm as text + pdfplumber for table) |
| **16** | Treatment table rows 3–8 + referral criteria | ✅ Extract |
| 17 | Documentation forms | **SKIP** |
| **18** | Tissue type illustrations (descriptions) | ✅ Extract |
| 19–33 | Administrative forms (appendices) | **SKIP** |
| **34** | Glossary (Glosari) — skip References section | ✅ Extract glossary only |
| 35–36 | Back matter | **SKIP** |

### Chunk architecture

| Chunk | Section | Source |
|---|---|---|
| 1 | T.I.M.E. Wound Assessment Principles | Page 14 blocks |
| 2 | Wound Assessment Algorithm (decision tree) | Page 15 blocks (reconstructed) |
| 3–10 | Wound Type 1–8 Treatment Recommendations | pdfplumber table (pages 15–16) |
| 11 | Cases Requiring Hospital Referral | Page 16 blocks |
| 12 | Tissue Type Illustrations & Descriptions | Page 18 blocks |
| 13 | Clinical Glossary | Page 34 blocks |

In [18]:
# ── CELL 1 · Dependencies & paths ─────────────────────────────────────────────
# Uncomment if any library is missing:
# !pip install pymupdf pdfplumber -q

import fitz          # PyMuPDF — native text layer extraction
import pdfplumber    # bordered table extraction
import re
import json
import hashlib
import unicodedata
import statistics
from pathlib import Path
from collections import defaultdict, Counter

# ── paths ──────────────────────────────────────────────────────────────────────
PDF_PATH    = "../clinical_pdfs_v2/Garis Panduan Perkhidmatan Penjagaan Luka di Fasiliti Kesihatan Primer.pdf"
SOURCE_NAME = "Garis Panduan Perkhidmatan Penjagaan Luka di Fasiliti Kesihatan Primer.pdf"
OUT_DIR     = Path("../ingestion_output_no_ai")
OUT_DIR.mkdir(exist_ok=True)

# ── page indices (0-based) to extract ──────────────────────────────────────────
PAGE_TIME       = 13   # PDF page 14 — T.I.M.E. assessment
PAGE_ALGO       = 14   # PDF page 15 — algorithm + treatment table rows 1-2
PAGE_TABLE2     = 15   # PDF page 16 — treatment table rows 3-8 + referral criteria
PAGE_ILLUS      = 17   # PDF page 18 — tissue illustrations
PAGE_GLOSSARY   = 33   # PDF page 34 — glossary

# ── constants ─────────────────────────────────────────────────────────────────
RUNNING_HEADER  = "GARIS PANDUAN PERKHIDMATAN PENJAGAAN LUKA DI FASILITI KESIHATAN PRIMER"
MIN_CHUNK_CHARS = 60

print("✅ imports ok")

✅ imports ok


## Step 1 · Helper — single-column block extraction

In [19]:
# ── CELL 2 · Block extraction helpers ─────────────────────────────────────────

def clean_block_text(text: str) -> str:
    """
    Clean a single text block from PyMuPDF:
    - Remove running page header
    - NFKC normalise (ligatures, private-use bullets \uf097 → •)
    - Strip lone page-number strings
    - Collapse whitespace
    """
    if RUNNING_HEADER in text:
        return ""
    text = unicodedata.normalize("NFKC", text)
    text = text.replace("\uf097", "•")       # private-use bullet used in this PDF
    text = text.strip()
    # Discard lone page numbers (1–2 digit strings)
    if re.fullmatch(r"\d{1,2}", text):
        return ""
    text = re.sub(r"[ \t]+", " ", text)      # collapse horizontal whitespace
    text = re.sub(r"\n{3,}", "\n\n", text)   # max 2 consecutive newlines
    return text.strip()


def get_page_blocks(doc: fitz.Document, pg_idx: int) -> list[dict]:
    """
    Return cleaned text blocks for one page, sorted by vertical position (y0).
    Image blocks (type != 0) are discarded.
    Returns list of {x0, y0, x1, y1, text}.
    """
    pg  = doc[pg_idx]
    raw = pg.get_text("blocks", sort=True)
    result = []
    for b in raw:
        if b[6] != 0:          # skip image/non-text blocks
            continue
        t = clean_block_text(b[4])
        if t:
            result.append({"x0": b[0], "y0": b[1],
                           "x1": b[2], "y1": b[3], "text": t})
    return result


def blocks_to_str(blocks: list[dict]) -> str:
    """Join block texts with newlines."""
    return "\n".join(b["text"] for b in blocks)


def cell_text(cell) -> str:
    """Clean and normalise a pdfplumber table cell value."""
    if cell is None:
        return ""
    return re.sub(r"\s+", " ", unicodedata.normalize("NFKC", str(cell))).strip()


def make_chunk_id(source: str, section: str, idx: int = 0) -> str:
    raw = f"{source}::{section}::{idx}"
    return hashlib.md5(raw.encode()).hexdigest()[:12]


# Verify we can open the PDF
doc = fitz.open(PDF_PATH)
print(f"✅ Opened PDF: {len(doc)} pages")
print(f"   Useful pages (0-indexed): {[PAGE_TIME, PAGE_ALGO, PAGE_TABLE2, PAGE_ILLUS, PAGE_GLOSSARY]}")

✅ Opened PDF: 36 pages
   Useful pages (0-indexed): [13, 14, 15, 17, 33]


## Step 2 · T.I.M.E. Wound Assessment Principles (page 14)

In [20]:
# ── CELL 3 · T.I.M.E. chunk ───────────────────────────────────────────────────
# Page 14 contains the T.I.M.E. framework as a diagram with scattered text blocks.
# We reconstruct it as a clean structured text rather than joining the raw blocks,
# because the block order doesn't follow reading order (the diagram has T/I/M/E
# columns that cross each other spatially).

time_raw_blocks = get_page_blocks(doc, PAGE_TIME)

print("Raw blocks on TIME page:")
for b in time_raw_blocks:
    print(f"  [{b['y0']:.0f}]  {repr(b['text'][:80])}")

# ── Build the clean structured chunk ──────────────────────────────────────────
TIME_CHUNK_TEXT = """PRINSIP ASAS PENILAIAN LUKA — T.I.M.E. Wound Bed Assessment Framework

Reference: *T.I.M.E. - Principles of Wound Bed Assessment And Preparation
(Wound Care Manual, First Edition 2014, Ministry of Health Malaysia, p.14)

Every wound assessment must evaluate three dimensions:

1. SIZE
   Measure wound surface area: Length (cm), Width (cm), Depth (cm)

2. SURROUNDING SKIN
   Assess for signs of: infection, ischaemia, maceration, crepitus

3. T.I.M.E. (four components):

   T — Tissue
       • Viable tissue: Granulation and new epithelial tissue
       • Non-viable tissue: Necrotic or slough tissue

   I — Infection / Inflammation
       • Signs and symptoms of infection:
         - Presence of pus
         - Pain / tenderness
         - Malodour

   M — Moisture imbalance
       • Exudate level assessment:
         - Dry / minimal exudate
         - Moderate / wet exudate

   E — Epidermal margin
       • Advancing wound edges  (good healing)
       • Non-advancing wound edges (stalled healing)

The T.I.M.E. assessment is the cornerstone of the wound type classification used in
the Garis Panduan wound care algorithm (see Wound Assessment Algorithm chunk)."""

print(f"\nT.I.M.E. chunk length: {len(TIME_CHUNK_TEXT)} chars")
print(TIME_CHUNK_TEXT[:500])

Raw blocks on TIME page:
  [57]  'PRINSIP ASAS PENILAIAN LUKA'
  [156]  'SIZE'
  [155]  'SURROUNDING\nSKIN'
  [172]  '*T.I.M.E.'
  [191]  'Measure wound\nsurface area\n(Length, width,\ndepth)'
  [206]  'Assess for signs of\ninfection/ ischaemia,\nmaceration, crepitus'
  [375]  'WOUND\nASSESSMENT'
  [472]  '*T.I.M.E. - Principles of Wound Bed Assessment And Preparation'
  [529]  'T : Tissue'
  [529]  'I : Infection/\nInflammation'
  [529]  'M : Moisture\nimbalance'
  [529]  'E : Epidermal\nmargin'
  [559]  '• Viable\nGranulation and new \nepithelial'
  [574]  '• Advancing\n• Non advancing'
  [574]  '• Exudate level dry/\nminimal or moderate/\nwet'
  [574]  '• Signs and symptoms\nof infection. Example ;\npresence of pus, pain,\nmalodour'
  [612]  '• Non viable\nNecrotic, slough tissue'
  [702]  'Rujukan : Wound Care Manual (First Edition, 2014), Ministry of Health, m/s 14.'

T.I.M.E. chunk length: 1171 chars
PRINSIP ASAS PENILAIAN LUKA — T.I.M.E. Wound Bed Assessment Framework

Reference

## Step 3 · Wound Assessment Algorithm (page 15 — decision tree)

In [21]:
# ── CELL 4 · Algorithm chunk ───────────────────────────────────────────────────
# The decision tree on page 15 is a visual diagram. Its text blocks give us
# the labels but not the logic flow. We reconstruct the decision logic explicitly
# so it is semantically useful for RAG retrieval.

ALGO_CHUNK_TEXT = """ASPEK PENJAGAAN LUKA — Wound Assessment Algorithm (Decision Tree)

This algorithm classifies wounds into 8 types using three sequential criteria.
It drives the selection of dressing material, antibiotics, and surgical procedures.

DECISION CRITERIA (applied in order):

Criterion 1 — TISSUE NECROSIS / SLOUGH
  • < 25% necrotic / slough tissue  →  Wound Types 1, 2, 3, 4
  • > 25% necrotic / slough tissue  →  Wound Types 5, 6, 7, 8

Criterion 2 — WOUND INFECTION (YES / NO)
  Within <25% necrosis group:
    • NO infection   →  Types 1, 2
    • YES infection  →  Types 3, 4
  Within >25% necrosis group:
    • NO infection   →  Types 5, 6
    • YES infection  →  Types 7, 8

Criterion 3 — MOISTURE / EXUDATE LEVEL
  • Dry / minimal exudate    →  Odd-numbered types  (1, 3, 5, 7)
  • Moderate / wet exudate   →  Even-numbered types (2, 4, 6, 8)

RESULTING WOUND TYPE MATRIX:

  Wound Type 1: < 25% necrosis | NO infection  | Dry / minimal   exudate
  Wound Type 2: < 25% necrosis | NO infection  | Moderate / wet  exudate
  Wound Type 3: < 25% necrosis | YES infection | Dry / minimal   exudate
  Wound Type 4: < 25% necrosis | YES infection | Moderate / wet  exudate
  Wound Type 5: > 25% necrosis | NO infection  | Dry / minimal   exudate
  Wound Type 6: > 25% necrosis | NO infection  | Moderate / wet  exudate
  Wound Type 7: > 25% necrosis | YES infection | Dry / minimal   exudate
  Wound Type 8: > 25% necrosis | YES infection | Moderate / wet  exudate

Each wound type has specific dressing, antibiotic and debridement recommendations.
See: Wound Types 1–8 Treatment Recommendation chunks.
Note: Wound types 6, 7, and 8 require referral to hospital."""

print(f"Algorithm chunk length: {len(ALGO_CHUNK_TEXT)} chars")
print(ALGO_CHUNK_TEXT[:500])

Algorithm chunk length: 1657 chars
ASPEK PENJAGAAN LUKA — Wound Assessment Algorithm (Decision Tree)

This algorithm classifies wounds into 8 types using three sequential criteria.
It drives the selection of dressing material, antibiotics, and surgical procedures.

DECISION CRITERIA (applied in order):

Criterion 1 — TISSUE NECROSIS / SLOUGH
  • < 25% necrotic / slough tissue  →  Wound Types 1, 2, 3, 4
  • > 25% necrotic / slough tissue  →  Wound Types 5, 6, 7, 8

Criterion 2 — WOUND INFECTION (YES / NO)
  Within <25% necrosis gr


## Step 4 · Treatment Recommendation Table — Wound Types 1–8 (pages 15–16)

The table spans two pages. pdfplumber detects the bordered cells reliably.
Each wound type becomes an independent chunk for precise retrieval.

In [22]:
# ── CELL 5 · Extract treatment table rows with pdfplumber ─────────────────────

# Canonical wound descriptions (supplements/corrects the raw PDF cell text)
WOUND_TYPE_DESCRIPTIONS = {
    "1": "Clean, healthy granulating wound — dry/minimal exudate, NO infection, <25% necrosis",
    "2": "Clean and wet wound — moderate/wet exudate, NO infection, <25% necrosis",
    "3": "Dry, infected wound with <25% slough/necrotic tissue (most likely vascular in origin)",
    "4": "Wet, infected wound with <25% slough/necrotic tissue",
    "5": "Dry, non-infected wound with >25% slough/necrotic tissue",
    "6": "Wet, non-infected wound with >25% slough/necrotic tissue",
    "7": "Dry, infected wound with >25% slough/necrotic tissue",
    "8": "Wet, infected wound with >25% slough/necrotic tissue",
}

# Extract all table rows across both pages
raw_treatment_rows: list[list] = []
with pdfplumber.open(PDF_PATH) as pdf:
    # Page 15 (0-indexed 14): wound types 1–2
    for tbl in pdf.pages[PAGE_ALGO].find_tables():
        for row in tbl.extract():
            if row and cell_text(row[0]).isdigit():
                raw_treatment_rows.append(row)
    # Page 16 (0-indexed 15): wound types 3–8
    for tbl in pdf.pages[PAGE_TABLE2].find_tables():
        for row in tbl.extract():
            if row and cell_text(row[0]).isdigit():
                raw_treatment_rows.append(row)

print(f"Treatment rows extracted: {len(raw_treatment_rows)}")
print("\nWound types found:", sorted(cell_text(r[0]) for r in raw_treatment_rows))

# Preview each row
for row in raw_treatment_rows:
    wt = cell_text(row[0])
    desc = cell_text(row[1])[:40]
    dress = cell_text(row[2])[:50]
    print(f"  Type {wt}: {desc!r} | dressing: {dress!r}")

Treatment rows extracted: 8

Wound types found: ['1', '2', '3', '4', '5', '6', '7', '8']
  Type 1: 'Clean, healthy granulating wound' | dressing: 'All types of dressing material except silver, char'
  Type 2: 'Clean and wet wound' | dressing: '1. Foam 2. Alginate 4. Hydrofiber 5. Polymeric mem'
  Type 3: 'Dry, infected wound with <25% slough/ ne' | dressing: '1.Tulle 2.Hydrogel 3.Hydrocolloid 4.Silver dressin'
  Type 4: 'Wet, infected wound with <25% slough/ ne' | dressing: '1.Alginate 2.Foam 3.Silver 4.Hydrofiber 5.Polymeri'
  Type 5: 'Dry, infected wound with >25% slough/ ne' | dressing: '1.Hydrogel 2.Hydrocolloid 3.Polymeric membrane'
  Type 6: 'Wet, non-infected wound with >25% slough' | dressing: '1. Alginate 2.Foam 3.Polymeric membrane 4.Hydrofib'
  Type 7: 'Dry, infected wound with >25% slough/ ne' | dressing: '1. Silver dressing 2.Hydrogel 3.Hydrocolloid 4.Iod'
  Type 8: 'Wet, infected wound with >25% slough/ ne' | dressing: '1.Alginate 2.Silver dressing 3.Hydrofiber 4.Foam 5'


In [23]:
# ── CELL 6 · Format one chunk per wound type ───────────────────────────────────

def format_treatment_chunk(row: list) -> str:
    """
    Convert one table row [wound_type, description, dressings, antibiotics, surgical]
    into a self-contained, richly annotated text chunk.
    """
    wt          = cell_text(row[0])
    description = cell_text(row[1])
    dressings   = cell_text(row[2])
    antibiotics = cell_text(row[3])
    surgical    = cell_text(row[4])

    # Use canonical description if available
    canonical_desc = WOUND_TYPE_DESCRIPTIONS.get(wt, description)

    # Determine wound category for context
    necrosis = "<25%" if wt in ("1","2","3","4") else ">25%"
    infected = "YES"  if wt in ("3","4","7","8") else "NO"
    moisture = "dry/minimal" if wt in ("1","3","5","7") else "moderate/wet"
    needs_referral = "YES — referral to hospital recommended" if wt in ("6","7","8") else "No"

    lines = [
        f"RAWATAN YANG DISARANKAN (Treatment Recommendation) — Wound Type {wt}",
        f"",
        f"Clinical Context (from T.I.M.E. algorithm):",
        f"  • Necrosis/slough: {necrosis} of wound surface",
        f"  • Wound infection: {infected}",
        f"  • Moisture/exudate: {moisture}",
        f"  • Requires hospital referral: {needs_referral}",
        f"",
        f"Wound Type {wt} Description:",
        f"  {canonical_desc}",
        f"",
        f"Recommended Dressing Materials:",
        f"  {dressings}",
        f"",
        f"Antibiotic Recommendation:",
        f"  {antibiotics}",
        f"",
        f"Surgical / Procedural Recommendation:",
        f"  {surgical}",
        f"",
        f"(Reference: Garis Panduan Perkhidmatan Penjagaan Luka di Fasiliti Kesihatan Primer,",
        f" Ministry of Health Malaysia 2019; based on Wound Care Manual 2014, pp.164–167)",
    ]
    return "\n".join(lines)


# Generate one text block per wound type
treatment_chunk_texts: dict[str, str] = {}
for row in raw_treatment_rows:
    wt = cell_text(row[0])
    if wt.isdigit() and len(row) >= 5:
        treatment_chunk_texts[wt] = format_treatment_chunk(row)

print(f"Treatment chunks built: {len(treatment_chunk_texts)} (wound types {sorted(treatment_chunk_texts.keys())})")

# Print full preview of wound types 1 and 8 as spot-check
for wt in ["1", "8"]:
    print(f"\n{'='*60}\nWound Type {wt}\n{'='*60}")
    print(treatment_chunk_texts[wt])

Treatment chunks built: 8 (wound types ['1', '2', '3', '4', '5', '6', '7', '8'])

Wound Type 1
RAWATAN YANG DISARANKAN (Treatment Recommendation) — Wound Type 1

Clinical Context (from T.I.M.E. algorithm):
  • Necrosis/slough: <25% of wound surface
  • Wound infection: NO
  • Moisture/exudate: dry/minimal
  • Requires hospital referral: No

Wound Type 1 Description:
  Clean, healthy granulating wound — dry/minimal exudate, NO infection, <25% necrosis

Recommended Dressing Materials:
  All types of dressing material except silver, charcoal and special advanced dressing materials.

Antibiotic Recommendation:
  No

Surgical / Procedural Recommendation:
  1.Ready for secondary wound closure 2.If the wound is small, continue dressing till the wound heals by secondary intention 3.Frequency of wound dressing varies depending on type of wound and also dressing material used

(Reference: Garis Panduan Perkhidmatan Penjagaan Luka di Fasiliti Kesihatan Primer,
 Ministry of Health Malaysia 2019; b

## Step 5 · Cases Requiring Hospital Referral (page 16 bottom)

In [24]:
# ── CELL 7 · Referral criteria chunk ──────────────────────────────────────────
# The referral criteria appear in the lower half of page 16 (0-indexed 15),
# after the treatment table. We extract the specific block containing it.

referral_blocks = get_page_blocks(doc, PAGE_TABLE2)

# The referral section starts at the block containing 'DIRUJUK KE HOSPITAL'
referral_raw_texts = []
in_referral = False
for b in referral_blocks:
    if "DIRUJUK KE HOSPITAL" in b["text"] or "KES YANG PERLU" in b["text"]:
        in_referral = True
    if in_referral:
        referral_raw_texts.append(b["text"])

print("Raw referral blocks:")
for t in referral_raw_texts:
    print(f"  {repr(t[:100])}")

# Build clean referral chunk
# The block structure: header, three wound type image labels, then bullet criteria
REFERRAL_CHUNK_TEXT = """KES YANG PERLU DIRUJUK KE HOSPITAL ATAU LAIN-LAIN INSTITUSI
(Cases Requiring Referral to Hospital or Other Institution)

Wound types that require hospital referral:
  i.   Luka jenis 6 — Wet, non-infected wound with >25% slough/necrotic tissue
  ii.  Luka jenis 7 — Dry, infected wound with >25% slough/necrotic tissue
  iii. Luka jenis 8 — Wet, infected wound with >25% slough/necrotic tissue

Criteria for referral (any of the following):

  • Wound characteristics require extensive care such as:
    - Surgical debridement
    - Vacuum (negative pressure) dressing
    - Other advanced wound procedures

  • Patient has other acute complications.

  • Systemic wound complications such as:
    - Sepsis
    - Severe cellulitis

  • Co-morbid disease complications requiring inpatient management:
    - Heart failure
    - Renal failure
    - Other conditions requiring ward admission

  • Other conditions as determined by the State-Level Wound Care Committee
    (Jawatankuasa Penjagaan Luka Peringkat Negeri).

(Reference: Garis Panduan Perkhidmatan Penjagaan Luka di Fasiliti Kesihatan Primer,
 Ministry of Health Malaysia 2019, p.14)"""

print(f"\nReferral chunk length: {len(REFERRAL_CHUNK_TEXT)} chars")

Raw referral blocks:
  'KES YANG PERLU DIRUJUK KE HOSPITAL ATAU LAIN-LAIN\nINSTITUSI\n• Ciri-ciri luka :'
  'i. Luka jenis 6 (wet, non\ninfected wound with >25%\nslough/ necrotic tissue)'
  'ii. Luka jenis 7 (dry, infected\nwound with >25% slough/\nnecrotic tissue)'
  'iii. Luka jenis 8 (wet, infected\nwound with >25% slough/\nnecrotic tissue)'
  '• Memerlukan penjagaan luka yang extensive seperti surgical debridement, vacuum dressing dan \n lain-'

Referral chunk length: 1141 chars


## Step 6 · Tissue Type Illustrations & Descriptions (page 18)

In [25]:
# ── CELL 8 · Tissue illustrations chunk ───────────────────────────────────────
# Page 18 contains photograph labels and tissue-type descriptions.
# The images themselves cannot be embedded in text chunks, but the
# descriptive labels are clinically useful for retrieval.

illus_blocks = get_page_blocks(doc, PAGE_ILLUS)
print("Illustration page blocks:")
for b in illus_blocks:
    print(f"  {repr(b['text'][:100])}")

# Build structured tissue description chunk
ILLUS_CHUNK_TEXT = """PANDUAN ILUSTRASI JENIS LUKA — Tissue Type Visual Guide

This section describes the visual characteristics of wound tissue types used in
the T.I.M.E. wound assessment framework.

1. NECROTIC / SLOUGH TISSUE
   - Dead, non-viable tissue
   - May appear black (eschar/necrotic) or yellow/grey (slough)
   - Requires debridement before wound can progress to healing

2. GRANULATION TISSUE
   - New connective tissue formed during wound healing
   - Appearance: Red, cobblestone / beefy texture
   - Only present in full-thickness wounds
   - Indicates healthy wound progression

3. EPITHELIAL TISSUE
   - Regrowth of the epidermis (outermost skin layer)
   - Appearance: Pink or pearly colour, smooth and shiny
   - Indicates advanced wound healing / near closure

4. PRESENCE OF EXUDATES ON TISSUE
   - Exudate is wound fluid that may be serous, sanguineous, or purulent
   - Level assessed as: dry / minimal  OR  moderate / wet
   - Heavy exudate requires absorbent dressings (alginate, foam, hydrofiber)

5. PRESENCE OF INFECTION ON TISSUE
   - Signs: redness, warmth, swelling, pain, purulent discharge, malodour
   - Infected wounds require antimicrobial dressings (silver, iodine)
     and may require systemic antibiotics based on culture & sensitivity (C&S)

Reference: Wound Care Manual, Ministry of Health Malaysia, 2014."""

print(f"\nIllustration chunk length: {len(ILLUS_CHUNK_TEXT)} chars")

Illustration page blocks:
  'PANDUAN ILUSTRASI JENIS LUKA'
  'Necrotic/ Slough Tissue'
  'Granulation Tissue'
  '• Granulation Tissue\n- Red, cobblestone/ beefy\n- Only in full thickness \nwounds'
  'Epithelial Tissue'
  '• Epithelial Tissue\n- Regrowth of epidermis\n- Pink or pearly\n- Smooth, shiny'
  'Presence of Exudates On Tissue'
  'Presence of Infection On Tissue'
  'Rujuk : Wound Care Manual (2014)'

Illustration chunk length: 1328 chars


## Step 7 · Clinical Glossary (page 34 — Glosari section only)

In [26]:
# ── CELL 9 · Glossary chunk ───────────────────────────────────────────────────
# Page 34 contains RUJUKAN (references — skip) followed by Glosari.
# We extract only from 'Glosari' onwards.

glossary_blocks = get_page_blocks(doc, PAGE_GLOSSARY)
print("Glossary page blocks:")
for b in glossary_blocks:
    print(f"  [{b['y0']:.0f}]  {repr(b['text'][:120])}")

# Extract only from the 'Glosari' header downwards
glosari_texts = []
in_glosari = False
for b in glossary_blocks:
    if b["text"].strip().lower().startswith("glosari"):
        in_glosari = True
        continue   # skip the 'Glosari' header itself, include the entries
    if in_glosari and b["text"].strip():
        glosari_texts.append(b["text"].strip())

print("\nGlosari entries:")
for t in glosari_texts:
    print(f"  {repr(t[:120])}")

# Format as clean chunk
glossary_body = "\n\n".join(glosari_texts)
GLOSSARY_CHUNK_TEXT = """GLOSARI — Clinical Glossary (Wound Care Terms)

Source: Garis Panduan Perkhidmatan Penjagaan Luka di Fasiliti Kesihatan Primer,
Ministry of Health Malaysia 2019.

""" + glossary_body

print(f"\nGlossary chunk length: {len(GLOSSARY_CHUNK_TEXT)} chars")
print(GLOSSARY_CHUNK_TEXT[:600])

Glossary page blocks:
  [35]  'RUJUKAN'
  [73]  '1. Is the use of modern versus conventional wound dressings warranted after primary knee \n and hip arthroplasty? Results'
  [130]  '2. Effectiveness of Advanced versus Conventional Wound Dressings on Healing of Chronic \n Wounds: Systematic Review and M'
  [188]  '3. Advances in wound care. Wound Dressings and Comparative Effectiveness Data Aditya \n Sood,1,* Mark S. Granick,1 and Na'
  [231]  '4. Wound Care Manual, First Edition, 2014. Ministry of Health Malaysia'
  [260]  '5. Manual on Pressure Injury Prevention Care Bundle, Hospital Kuala Lumpur'
  [289]  '6. ASEAN Plus Guidelines Management of Diabetic Wounds.'
  [352]  'Glosari'
  [389]  '1. Wound - A wound is an injury to the integument or to the underlying structures; It is \n visible result of individual '
  [446]  '2. Ulcer - An interruption of continuity of an epithelial surface with an inflamed base.It \n is usually a result of an u'
  [490]  '3. Diabetic foot - Diabetic foot

In [27]:
doc.close()   # done with PyMuPDF
print("✅ PDF closed")

✅ PDF closed


## Step 8 · Assemble all chunks

In [28]:
# ── CELL 10 · Assemble chunk list ─────────────────────────────────────────────

def make_chunk(
    section: str,
    parent_section: str,
    text: str,
    chunk_index: int = 0,
) -> dict:
    return {
        "chunk_id":       make_chunk_id(SOURCE_NAME, section, chunk_index),
        "source":         SOURCE_NAME,
        "section":        section,
        "parent_section": parent_section,
        "chunk_index":    chunk_index,
        "char_count":     len(text),
        "text":           text,
        "ai_summary":     text,   # overwrite with LLM summary if ENABLE_AI_SUMMARY = True
    }


chunks: list[dict] = []

# ── 1. T.I.M.E. Assessment ────────────────────────────────────────────────────
chunks.append(make_chunk(
    section        = "Wound Assessment — T.I.M.E. Framework",
    parent_section = "Wound Assessment",
    text           = TIME_CHUNK_TEXT,
))

# ── 2. Algorithm ─────────────────────────────────────────────────────────────
chunks.append(make_chunk(
    section        = "Wound Assessment — Decision Algorithm",
    parent_section = "Wound Assessment",
    text           = ALGO_CHUNK_TEXT,
))

# ── 3–10. Treatment recommendations (one per wound type) ─────────────────────
for wt in sorted(treatment_chunk_texts.keys(), key=int):
    text = treatment_chunk_texts[wt]
    if len(text) >= MIN_CHUNK_CHARS:
        chunks.append(make_chunk(
            section        = f"Treatment Recommendation — Wound Type {wt}",
            parent_section = "Treatment Recommendations (Rawatan yang Disarankan)",
            text           = text,
        ))

# ── 11. Referral criteria ─────────────────────────────────────────────────────
chunks.append(make_chunk(
    section        = "Cases Requiring Hospital Referral",
    parent_section = "Clinical Management",
    text           = REFERRAL_CHUNK_TEXT,
))

# ── 12. Tissue illustrations ──────────────────────────────────────────────────
chunks.append(make_chunk(
    section        = "Tissue Type Descriptions — Visual Guide",
    parent_section = "Wound Assessment",
    text           = ILLUS_CHUNK_TEXT,
))

# ── 13. Glossary ──────────────────────────────────────────────────────────────
if len(GLOSSARY_CHUNK_TEXT) >= MIN_CHUNK_CHARS:
    chunks.append(make_chunk(
        section        = "Clinical Glossary",
        parent_section = "Reference",
        text           = GLOSSARY_CHUNK_TEXT,
    ))

print(f"Total chunks assembled: {len(chunks)}")
for c in chunks:
    print(f"  {c['section']!r:55s}  chars={c['char_count']:5d}")

Total chunks assembled: 13
  'Wound Assessment — T.I.M.E. Framework'                  chars= 1171
  'Wound Assessment — Decision Algorithm'                  chars= 1657
  'Treatment Recommendation — Wound Type 1'                chars=  948
  'Treatment Recommendation — Wound Type 2'                chars=  781
  'Treatment Recommendation — Wound Type 3'                chars=  780
  'Treatment Recommendation — Wound Type 4'                chars=  754
  'Treatment Recommendation — Wound Type 5'                chars=  678
  'Treatment Recommendation — Wound Type 6'                chars=  823
  'Treatment Recommendation — Wound Type 7'                chars=  825
  'Treatment Recommendation — Wound Type 8'                chars=  874
  'Cases Requiring Hospital Referral'                      chars= 1141
  'Tissue Type Descriptions — Visual Guide'                chars= 1328
  'Clinical Glossary'                                      chars= 1325


## Step 9 · Quality validation

In [29]:
# ── CELL 11 · Quality checks ──────────────────────────────────────────────────

all_combined = " ".join(c["text"].lower() for c in chunks)

MUST_CONTAIN = [
    ("t.i.m.e.",          "T.I.M.E. framework"),
    ("granulation",        "Granulation tissue"),
    ("necrotic",           "Necrotic tissue"),
    ("wound type 1",       "Wound Type 1"),
    ("wound type 8",       "Wound Type 8"),
    ("alginate",           "Alginate dressing"),
    ("silver dressing",    "Silver dressing"),
    ("hydrofiber",         "Hydrofiber"),
    ("hydrogel",           "Hydrogel"),
    ("foam",               "Foam dressing"),
    ("iodine",             "Iodine dressing"),
    ("charcoal",           "Charcoal dressing (Type 8)"),
    ("debridement",        "Debridement guidance"),
    ("hospital",           "Referral criteria"),
    ("epithelial",         "Epithelial tissue"),
    ("debridement is the", "Glossary — debridement"),
]

print("═" * 70)
print("QUALITY REPORT — GP Wound Care Guidelines")
print("═" * 70)

print(f"\n📦 Total chunks    : {len(chunks)}")
char_counts = [c["char_count"] for c in chunks]
print(f"   Chars — min    : {min(char_counts)}")
print(f"   Chars — mean   : {statistics.mean(char_counts):.0f}")
print(f"   Chars — max    : {max(char_counts)}")

# Duplicate check
seen, dupes = {}, []
for c in chunks:
    key = c["text"][:150]
    if key in seen:
        dupes.append((seen[key], c["chunk_id"]))
    else:
        seen[key] = c["chunk_id"]
print(f"\n🔁 Duplicates      : {len(dupes)}")

print("\n✅ Content coverage:")
for kw, label in MUST_CONTAIN:
    found = kw in all_combined
    print(f"   {'✓' if found else '✗ MISSING'} {label}")

print("\n🗂  Chunks by parent section:")
parent_counts = Counter(c["parent_section"] for c in chunks)
for sec, cnt in sorted(parent_counts.items(), key=lambda x: -x[1]):
    print(f"   {cnt:2d} × {sec!r}")

══════════════════════════════════════════════════════════════════════
QUALITY REPORT — GP Wound Care Guidelines
══════════════════════════════════════════════════════════════════════

📦 Total chunks    : 13
   Chars — min    : 678
   Chars — mean   : 1007
   Chars — max    : 1657

🔁 Duplicates      : 0

✅ Content coverage:
   ✓ T.I.M.E. framework
   ✓ Granulation tissue
   ✓ Necrotic tissue
   ✓ Wound Type 1
   ✓ Wound Type 8
   ✓ Alginate dressing
   ✓ Silver dressing
   ✓ Hydrofiber
   ✓ Hydrogel
   ✓ Foam dressing
   ✓ Iodine dressing
   ✓ Charcoal dressing (Type 8)
   ✓ Debridement guidance
   ✓ Referral criteria
   ✓ Epithelial tissue
   ✓ Glossary — debridement

🗂  Chunks by parent section:
    8 × 'Treatment Recommendations (Rawatan yang Disarankan)'
    3 × 'Wound Assessment'
    1 × 'Clinical Management'
    1 × 'Reference'


In [30]:
# ── CELL 12 · Spot-check individual chunks ────────────────────────────────────

def preview_chunk(idx: int):
    c = chunks[idx]
    print(f"\n{'─'*65}")
    print(f"[{idx}] chunk_id    : {c['chunk_id']}")
    print(f"     section      : {c['section']}")
    print(f"     parent       : {c['parent_section']}")
    print(f"     chars        : {c['char_count']}")
    print(f"TEXT:")
    print(c["text"][:700])
    if len(c["text"]) > 700:
        print("... [truncated]")

# Spot-check: TIME, algorithm, wound type 4 (wet+infected+<25%), wound type 7 (dry+infected+>25%)
for idx in [0, 1, 4, 8, -2, -1]:
    preview_chunk(idx)


─────────────────────────────────────────────────────────────────
[0] chunk_id    : 8409dedeea26
     section      : Wound Assessment — T.I.M.E. Framework
     parent       : Wound Assessment
     chars        : 1171
TEXT:
PRINSIP ASAS PENILAIAN LUKA — T.I.M.E. Wound Bed Assessment Framework

Reference: *T.I.M.E. - Principles of Wound Bed Assessment And Preparation
(Wound Care Manual, First Edition 2014, Ministry of Health Malaysia, p.14)

Every wound assessment must evaluate three dimensions:

1. SIZE
   Measure wound surface area: Length (cm), Width (cm), Depth (cm)

2. SURROUNDING SKIN
   Assess for signs of: infection, ischaemia, maceration, crepitus

3. T.I.M.E. (four components):

   T — Tissue
       • Viable tissue: Granulation and new epithelial tissue
       • Non-viable tissue: Necrotic or slough tissue

   I — Infection / Inflammation
       • Signs and symptoms of infection:
         - Presence of
... [truncated]

──────────────────────────────────────────────────────────

## Step 10 · (Optional) LLM ai_summary enrichment

In [31]:
# ── CELL 13 · LLM ai_summary (optional) ───────────────────────────────────────
ENABLE_AI_SUMMARY = False   # ← set True when OpenAI key is available

if ENABLE_AI_SUMMARY:
    import os
    from openai import OpenAI
    client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

    SYSTEM_PROMPT = (
        "You are a medical summarisation assistant. "
        "Rewrite the following wound-care guideline text as a clear, complete, self-contained "
        "clinical summary suitable for retrieval-augmented generation. "
        "Preserve all clinical facts, dressing names, wound types, indications, and "
        "contraindications. Return only the summary text — no preamble."
    )

    print("Running AI summaries...")
    for i, c in enumerate(chunks):
        print(f"  [{i+1}/{len(chunks)}] {c['section']}")
        resp = client.chat.completions.create(
            model="gpt-4o-mini",
            temperature=0,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user",   "content": c["text"]},
            ],
        )
        c["ai_summary"] = resp.choices[0].message.content.strip()
    print("✅ AI summaries done")
else:
    print("ℹ️  AI summary disabled — ai_summary == text (raw chunk)")

ℹ️  AI summary disabled — ai_summary == text (raw chunk)


## Step 11 · Export JSON

In [32]:
# ── CELL 14 · Export ChromaDB-ready JSON ──────────────────────────────────────
# Format mirrors SFP_wound_dressings_kept.json so the general ingestion notebook
# can load all PDFs' outputs uniformly.

output = {
    "meta": {
        "total_chunks":    len(chunks),
        "kept_count":      len(chunks),
        "ai_summarised":   sum(1 for c in chunks if c["ai_summary"] != c["text"]),
        "extraction":      "PyMuPDF native text blocks + pdfplumber tables (pages 15-16)",
        "chunking":        "manual section-aware — one chunk per logical section / wound type",
        "pages_used":      [14, 15, 16, 18, 34],
        "pages_skipped":   "1-13, 17, 19-33, 35-36 (admin, SOP, forms, appendices)",
        "chunk_params": {
            "min_characters": MIN_CHUNK_CHARS,
        },
        "note": "Use ai_summary field for reference_contexts and ChromaDB page_content."
    },
    "kept_ids_by_source": {
        SOURCE_NAME: [c["chunk_id"] for c in chunks]
    },
    "kept_chunks": [
        {
            "chunk_id":       c["chunk_id"],
            "source":         c["source"],
            "section":        c["section"],
            "parent_section": c["parent_section"],
            "chunk_index":    c["chunk_index"],
            "char_count":     c["char_count"],
            "text":           c["text"],
            "ai_summary":     c["ai_summary"],
        }
        for c in chunks
    ]
}

out_path = OUT_DIR / "GP_wound_dressings_kept.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(output, f, ensure_ascii=False, indent=2)

print(f"✅ Exported {len(chunks)} chunks → {out_path}")
print(f"   File size: {out_path.stat().st_size / 1024:.1f} KB")

✅ Exported 13 chunks → ..\ingestion_output_no_ai\GP_wound_dressings_kept.json
   File size: 32.2 KB


In [33]:
# ── CELL 15 · Final summary table ─────────────────────────────────────────────

with open(OUT_DIR / "GP_wound_dressings_kept.json") as f:
    exported = json.load(f)

print("═" * 72)
print("INGESTION COMPLETE — GP Wound Care Guidelines (MOH Malaysia 2019)")
print("═" * 72)

hdr = f"{'#':>3}  {'Chunk ID':14}  {'Section':55}  {'Chars':>5}  {'AI?':8}"
print(hdr)
print("-" * len(hdr))
for i, c in enumerate(exported["kept_chunks"], 1):
    ai = "yes" if c["ai_summary"] != c["text"] else "no (raw)"
    print(f"{i:3d}  {c['chunk_id']:14}  {c['section'][:55]:55s}  {c['char_count']:5d}  {ai}")

print()
print(f"Output JSON : {OUT_DIR / 'GP_wound_dressings_kept.json'}")
print(f"Next steps  :")
print(f"  1) Review chunk content — edit TIME/ALGO/ILLUS/REFERRAL text constants if needed")
print(f"  2) Set ENABLE_AI_SUMMARY = True to enrich ai_summary with GPT")
print(f"  3) Load into vector store via general_ingestion notebook")

# RAGAS reference-context lookup
ref_ctx_by_section = defaultdict(list)
for c in exported["kept_chunks"]:
    ref_ctx_by_section[c["section"]].append(c["ai_summary"])

print(f"\nRAGAS reference_context lookup ready — {len(ref_ctx_by_section)} sections")

════════════════════════════════════════════════════════════════════════
INGESTION COMPLETE — GP Wound Care Guidelines (MOH Malaysia 2019)
════════════════════════════════════════════════════════════════════════
  #  Chunk ID        Section                                                  Chars  AI?     
---------------------------------------------------------------------------------------------
  1  8409dedeea26    Wound Assessment â€” T.I.M.E. Framework                   1171  no (raw)
  2  bd2bb8e1321e    Wound Assessment â€” Decision Algorithm                   1657  no (raw)
  3  52ef696853c7    Treatment Recommendation â€” Wound Type 1                  948  no (raw)
  4  4643f10b8894    Treatment Recommendation â€” Wound Type 2                  781  no (raw)
  5  c0a350e36ecf    Treatment Recommendation â€” Wound Type 3                  780  no (raw)
  6  d622ee9f4c9c    Treatment Recommendation â€” Wound Type 4                  754  no (raw)
  7  aad7a40107b0    Treatment Recom